<a href="https://colab.research.google.com/github/Manav010203/Quiz/blob/cloudmain/going_modular_first_Time_tinyvgg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import requests
import zipfile
from pathlib import Path

data_path = Path("data")
image_path = data_path / "pizza_steak_sushi"


# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Download pizza, steak, sushi data
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

# Unzip pizza, steak, sushi data
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)

# Remove zip file
os.remove(data_path / "pizza_steak_sushi.zip")

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


In [1]:
# Import modules required for train.py
import os
import torch
# import data_setup, engine, model_builder, utils

from torchvision import transforms

In [6]:
import os

# Create the directory if it doesn't exist
if not os.path.exists('going_modular'):
    os.makedirs('going_modular')
    print("Created 'going_modular' directory.")
else:
    print("'going_modular' directory already exists.")

Created 'going_modular' directory.


In [12]:
%%writefile going_modular/data_setup.py

import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()
def create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers):
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)
    class_names = train_data

    train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS)
    test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

    return train_dataloader, test_dataloader, class_names

Writing going_modular/data_setup.py


In [25]:
from going_modular import data_setup
train_dataloaders, test_dataloader , classe_names = going_modular.data_setup.create_dataloaders(train_dir,test_dir,transforms = transforms.Compose(),batch_size=32,num_workers=os.cpu_count())

NameError: name 'going_modular' is not defined

In [8]:
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir,test_dir

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

In [14]:
%%writefile going_modular/model_builder.py

import torch
from torch import nn

class TinyVGG(nn.Module):
  def __init__(self,input_shape:int,hidden_units:int,output_shape:int)-> None:
    super().__init__()
    self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,kernel_size=3,padding=0,stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=0,stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.layer2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=0,stride=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=0,stride=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*13*13,out_features=output_shape)
    )
  def forward(self,x):
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.classifier(x)
    return x

Writing going_modular/model_builder.py


In [17]:
import torch
from going_modular import model_builder
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
model = model_builder.TinyVGG(input_shape=3,hidden_units=10,output_shape=len(class_names)).to(device)

NameError: name 'class_name' is not defined

In [43]:
%%writefile going_modular/engine.py

import torch

from tqdm.auto import tqdm
from typing import Dict, List ,Tuple

def train_step(dataloader:torch.utils.data.DataLoader,
               model: torch.nn.Module,
               loss_function:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               device:torch.device)->Tuple[float,float]:

              model.train()
              train_loss,train_acc =0,0
              for batch,(X,y) in enumerate(dataloader):
                  X = X.to(device)
                  y = y.to(device)
                  y_pred = model(X)
                  loss =loss_function(y_pred,y)
                  train_loss+=loss.item()
                  optimizer.zero_grad()
                  loss.backward()
                  optimizer.step()
                  y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
                  train_acc += (y_pred_class == y).sum().item()/len(y_pred)


              train_loss = train_loss / len(dataloader)
              train_acc = train_acc/ len(dataloader)
              return train_loss,train_acc

def test_step(dataloader:torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_function:torch.nn.Module,
              device:torch.device)-> Tuple[float,float]:
            model.eval()
            test_loss,test_Acc =0,0
            for batch,(X,y) in enumerate(dataloader):
                  X=X.to(device)
                  y=y.to(device)
                  y_pred = model(X)
                  loss = loss_function(y_pred,y)
                  test_loss+=loss.item()
                  y_pred_class = y_pred.argmax(dim=1)
                  test_Acc += (y_pred_class == y).sum().item()/len(y_pred)

            test_loss = test_loss /len(dataloader)
            test_Acc = test_Acc/len(dataloader)
            return test_loss,test_Acc

def train(train_dataloader:torch.utils.data.DataLoader,
          test_dataloader:torch.utils.data.DataLoader,
          model: torch.nn.Module,
          optimizer:torch.optim.Optimizer,
          loss_function:torch.nn.Module,
          epochs:int,
          device:torch.device)->Dict[str,List]:
          results = {"train_loss":[],
                     "train_acc":[],
                     "test_loss":[],
                     "test_acc":[]
                     }

          for epoch in tqdm(range(epochs)):
              train_loss,train_acc = train_step(dataloader=train_dataloader,model=model,loss_function=loss_function,optimizer=optimizer,device=device)
              test_loss,test_acc = test_step(dataloader=test_dataloader,model=model,loss_function=loss_function,device=device)


              print(
                  f"Epoch: {epoch+1} | "
                  f"train_loss: {train_loss:.4f} | "
                  f"train_acc: {train_acc:.4f} | "
                  f"test_loss: {test_loss:.4f} | "
                  f"test_acc: {test_acc:.4f}"
              )
              results["train_loss"].append(train_loss)
              results["train_acc"].append(train_acc)
              results["test_loss"].append(test_loss)
              results["test_acc"].append(test_acc)

          return results





Overwriting going_modular/engine.py


In [26]:
from going_modular import engine
engine.train()

TypeError: train() missing 7 required positional arguments: 'train_dataloader', 'test_dataloader', 'model', 'optimizer', 'loss_function', 'epochs', and 'device'

In [27]:
%%writefile going_modular/utils.py
import torch
from pathlib import Path

def save_model(model:torch.nn.Module,
               target_dir:str,
               model_name:str):
  model_path = Path(target_dir)
  model_path.mkdir(parents=True,exist_ok=True)
  assert model_name.endswith(".pth") or model_name.endswith(".pt"),"model name ends with pth or pt"
  model_save_path = model_path/model_name

  print(f"[INFO] savind model to :{model_save_path}")
  torch.save(obj=model.state_dict(),f=model_save_path)

Writing going_modular/utils.py


In [38]:
%%writefile going_modular/train.py
import torch
import os
from torchvision import transforms
import data_setup,engine,model_builder,utils

BATCH_SIZE = 32
EPOCHS = 5
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

device = "cuda" if torch.cuda.is_available() else "cpu"

data_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    # transforms.ToPILImage()
])

train_dataloader,test_dataloader,class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE,
    num_workers=os.cpu_count()
)
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)

engine.train(train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             model=model,
             optimizer=optimizer,
             loss_function=loss_fn,
             epochs=EPOCHS,
             device=device)

utils.save_model(model=model,
                 target_dir = "models",
                 model_name = "05_going_modular_script_model_tinyvgg_model.pth")




Overwriting going_modular/train.py


In [44]:
!python going_modular/train.py --model modular_model --batch_size BATCH_SIZE --lr LEARNING_RATE --num_epochs NUM_EPOCHS

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 4.3126 | train_acc: 0.3672 | test_loss: 1.9596 | test_acc: 0.1979
 20% 1/5 [00:01<00:07,  1.82s/it]Epoch: 2 | train_loss: 1.2778 | train_acc: 0.2695 | test_loss: 1.1513 | test_acc: 0.1875
 40% 2/5 [00:03<00:05,  1.94s/it]Epoch: 3 | train_loss: 1.1538 | train_acc: 0.2812 | test_loss: 0.9820 | test_acc: 0.5417
 60% 3/5 [00:06<00:04,  2.42s/it]Epoch: 4 | train_loss: 1.2998 | train_acc: 0.3203 | test_loss: 1.4127 | test_acc: 0.3750
 80% 4/5 [00:08<00:02,  2.21s/it]Epoch: 5 | train_loss: 1.0409 | train_acc: 0.5273 | test_loss: 0.9652 | test_acc: 0.5729
100% 5/5 [00:10<00:00,  2.12s/it]
[INFO] savind model to :models/05_going_modular_script_model_tinyvgg_model.pth


In [34]:
!ls /content/

data  going_modular  sample_data
